# Market Data Wrangling and Visualization (Identifying whales, $\Delta$ price, etc)

## Set Up

In [60]:
import time
import requests
import pandas as pd
import altair as alt

In [65]:
market_test = pd.read_csv("market_csvs/trumps-31_trades.csv")
market_test.drop(columns = ["name", "pseudonym", "bio", "profileImage", "profileImageOptimized", "transactionHash"])
market_test.head()

,proxyWallet,side,asset,conditionId,size,price,timestamp,title,slug,icon,eventSlug,outcome,outcomeIndex,name,pseudonym,bio,profileImage,profileImageOptimized,transactionHash
0,0xec4e8e525969c4b9072eed0193e587b73876d9db,SELL,1818328691843486982444167599079424376546601093...,0x775e71b9c9fd3c0392d32745839c2aedbb51fd329e37...,360.400,0.999000,1769930976,Trump's travel ban removed by January 31?,trumps-travel-ban-removed-by-january-31,https://polymarket-upload.s3.us-east-2.amazona...,trumps-travel-ban-removed-by-january-31,No,1,southbets,Meaty-Locket,Nothing to put here,NaN,NaN,0x103fb0be4e3bb3e3f3b0d5fba512c24e4d35a9592b5e...
1,0x5f94ea09f1fe4b506a785ac4526e27a9391f1454,SELL,1818328691843486982444167599079424376546601093...,0x775e71b9c9fd3c0392d32745839c2aedbb51fd329e37...,2.000,0.999000,1769929438,Trump's travel ban removed by January 31?,trumps-travel-ban-removed-by-january-31,https://polymarket-upload.s3.us-east-2.amazona...,trumps-travel-ban-removed-by-january-31,No,1,lighter-nls,Yummy-Randomization,NaN,NaN,NaN,0x1dd3603d32e098b08a70769fc9c62b07ed60d3a8bae3...
2,0x9380db17dd5ab2ce55301effa758fd71a6e501ed,BUY,1818328691843486982444167599079424376546601093...,0x775e71b9c9fd3c0392d32745839c2aedbb51fd329e37...,174.000,0.996731,1769890322,Trump's travel ban removed by January 31?,trumps-travel-ban-removed-by-january-31,https://polymarket-upload.s3.us-east-2.amazona...,trumps-travel-ban-removed-by-january-31,No,1,0x9380dB17dD5Ab2ce55301effA758fD71a6e501ed-176...,Old-Fashioned-Bullfighter,NaN,NaN,NaN,0xcf3a9c2592487800f66f2809e396874fd656e5849ab5...
3,0x89f0a333cda79fccc920dc0f003823e9bece1e62,BUY,1818328691843486982444167599079424376546601093...,0x775e71b9c9fd3c0392d32745839c2aedbb51fd329e37...,3.006,0.998000,1769858808,Trump's travel ban removed by January 31?,trumps-travel-ban-removed-by-january-31,https://polymarket-upload.s3.us-east-2.amazona...,trumps-travel-ban-removed-by-january-31,No,1,dissk,Giant-Innocence,NaN,NaN,NaN,0x68c581a0729c89a66bd626ec5b97ed6199a175679597...
4,0x272df07434614e735c7896b0eb9d3d26c78d9e36,BUY,1818328691843486982444167599079424376546601093...,0x775e71b9c9fd3c0392d32745839c2aedbb51fd329e37...,3.006,0.998000,1769858798,Trump's travel ban removed by January 31?,trumps-travel-ban-removed-by-january-31,https://polymarket-upload.s3.us-east-2.amazona...,trumps-travel-ban-removed-by-january-31,No,1,hjss,Showy-Secretary,NaN,NaN,NaN,0x5a22797ab118a6476187b81a883e42e2026effeb7cfe...


## Identifying Whales

In [66]:
# wallet statistic summary
wallet_stats = market_test.groupby("proxyWallet").agg(
    total_volume=("size", "sum"),
    n_trades=("size", "count"),
    avg_price=("price", "mean"),
    largest_trade=("size", "max"),
    first_trade=("timestamp", "min"),
    last_trade=("timestamp", "max")
)

# sort by size
wallet_stats = wallet_stats.sort_values("total_volume", ascending=False)
wallet_stats = wallet_stats.reset_index()
wallet_stats

# define thresholds for whales
market_volume = market_test["size"].sum()
volume_threshold = wallet_stats["total_volume"].quantile(0.95)
trade_threshold = wallet_stats["largest_trade"].quantile(0.95)


# whales can take two types: single traes and multiple trades, for now sort both
whales_df = wallet_stats[
    (wallet_stats["total_volume"] > volume_threshold) |
    (wallet_stats["largest_trade"] > trade_threshold)
]

print(volume_threshold, trade_threshold)
whales_df

434.0 326.0


,proxyWallet,total_volume,n_trades,avg_price,largest_trade,first_trade,last_trade
0,0x977449de8b5be74e089f3b28f035ebf3e5f46d3c,1510.000000,3,0.870000,755.0000,1766007929,1766395265
1,0xec4e8e525969c4b9072eed0193e587b73876d9db,1040.804887,8,0.941401,360.4000,1766264775,1769930976
2,0xd14aa86fa6cac2e09edd9944f5c0c0d48634a8ad,927.877100,1,0.969956,927.8771,1769232200,1769232200
3,0x60ccc2ea4a54b191b6114fa769c03359bb1af23f,669.000000,1,0.150000,669.0000,1766058414,1766058414
4,0xeaf1f1805537078c833c2acba571b1dd765668ca,635.000000,3,0.870000,326.0000,1766006877,1766395441
9,0xc8ab97a9089a9ff7e6ef0688e6e591a066946418,387.400000,2,0.043210,386.3100,1767440993,1769248852


## Plotting Price / Time

In [67]:
df = market_test.copy()

# convert price to be measured only for winning outcome (index = 0)
df["price_outcome"] = df["price"]
df.loc[df["outcomeIndex"] == 0, "price_outcome"] = (1 - df.loc[df["outcomeIndex"] == 0, "price"])

# parse time + clean
df = df.sort_values("timestamp")
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce")
df = df.dropna(subset=["timestamp", "price_outcome"]).sort_values("timestamp")

# plot graph
market_graph = alt.Chart(df).mark_line().encode(
    alt.X("timestamp:T").title("Time"),
    alt.Y("price_outcome:Q").title("Price (USD)")
).properties(width = 600, height = 300, title="$ over time of Resolution Outcome")

market_graph

alt.Chart(...)

## Watching Whales

In [68]:
df["is_whale"] = df["proxyWallet"].isin(whales_df["proxyWallet"])

whale_points = alt.Chart(df[df["is_whale"]]).mark_point(filled=True, size=60, color="Red").encode(
    alt.X("timestamp:T").title("Time"),
    alt.Y("price_outcome:Q").title("Price (USD)"),
    tooltip=[alt.Tooltip("proxyWallet:N", title="Wallet ID:"),
             alt.Tooltip("outcomeIndex:Q", title="Outcome:"),
             alt.Tooltip("side:N", title="Action:")]
)

chart = (market_graph + whale_points).properties(width = 600, height = 300, title="$ over time of Resolution Outcome with Whale Transactions")


chart

alt.LayerChart(...)

![Whale Watching](other/whalewatching.jpg)

This also works! some parts are a little funky and will have to clean up the visualization, time to move on to making all of this work coherently and do this for every market!